# MobileViT: Fine-tune a pretrained model for image classification

**Based on:** [Sayak Paul](https://twitter.com/RisingSayak), Keras MobileViT example<br>
**Description:** Fine-tune Hugging Face [`apple/mobilevit-small`](https://huggingface.co/apple/mobilevit-small) (ImageNet weights) on a downstream dataset instead of training a small model from scratch.

## Introduction

[MobileViT](https://arxiv.org/abs/2110.02178) ([Mehta et al.](https://arxiv.org/abs/2110.02178)) mixes convolutions and Transformers for efficient image classification. Here we **fine-tune** the official small checkpoint from the Hugging Face Hub rather than defining the architecture in Keras and training from random initialization.

**Requirements:** PyTorch (`torch`), `transformers`, `datasets`, and `accelerate` (for `Trainer`). Install if needed, for example: `pip install torch transformers datasets accelerate torchvision pillow evaluate`.

**Note:** In current `transformers` releases, image preprocessing is exposed as `AutoImageProcessor` / `MobileViTImageProcessor`; the older name `MobileViTFeatureExtractor` may not import. The notebook uses `AutoImageProcessor.from_pretrained("apple/mobilevit-small")`, which loads the same preprocessor config.

**Fine-tuning vs full training:** We load pretrained weights, replace the ImageNet classification head with `num_labels` for the task, and train with a **lower learning rate** than typical scratch training.

## Imports

In [2]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoImageProcessor,
    MobileViTForImageClassification,
    Trainer,
    TrainingArguments,
)

feature_extractor = AutoImageProcessor.from_pretrained("apple/mobilevit-small")

c:\Users\lucas\Documents\MIA\VPC_II\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


## Hyperparameters

In [3]:
# Fine-tuning defaults (smaller LR than scratch training).
batch_size = 16
epochs = 5
learning_rate = 5e-5
weight_decay = 0.01

_sz = getattr(feature_extractor, "size", {}) or {}
if isinstance(_sz, dict):
    image_size = int(_sz.get("height", _sz.get("shortest_edge", 256)))
else:
    image_size = 256


## Dataset preparation

We use [`tf_flowers`](https://www.tensorflow.org/datasets/catalog/tf_flowers) via Hugging Face `datasets`. Images are resized and normalized by `MobileViTFeatureExtractor` (same preprocessing as pretraining).


In [4]:
import io
import os
import tarfile
import urllib.request
from pathlib import Path

from datasets import Dataset, ClassLabel
from PIL import Image as PILImage

# Same archive as `tensorflow_datasets` tf_flowers — no TensorFlow / gcsfs needed.
_FLOWERS_URL = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
_CACHE_DIR = Path(os.environ.get("HF_HOME", Path.home() / ".cache")) / "tf_flowers_raw"
_ARCHIVE = _CACHE_DIR / "flower_photos.tgz"

label_names = ["dandelion", "daisy", "tulips", "sunflowers", "roses"]


def _ensure_flowers_archive():
    _CACHE_DIR.mkdir(parents=True, exist_ok=True)
    if not _ARCHIVE.is_file():
        print("Downloading flower_photos.tgz (same source as TFDS tf_flowers)...")
        urllib.request.urlretrieve(_FLOWERS_URL, _ARCHIVE)


def _load_sorted_image_label_pairs():
    _ensure_flowers_archive()
    records = []
    with tarfile.open(_ARCHIVE, "r:gz") as tar:
        for m in tar.getmembers():
            if not m.isfile() or not m.name.lower().endswith(".jpg"):
                continue
            parts = m.name.replace("\\", "/").split("/")
            if len(parts) < 2:
                continue
            folder = parts[-2].lower()
            if folder not in label_names:
                continue
            f = tar.extractfile(m)
            if f is None:
                continue
            img = PILImage.open(io.BytesIO(f.read())).copy()
            records.append((m.name, img, label_names.index(folder)))
    records.sort(key=lambda r: r[0])
    images = [r[1] for r in records]
    labels = [r[2] for r in records]
    return images, labels


def _pairs_to_hf(images, labels):
    ds = Dataset.from_dict({"image": images, "label": labels})
    return ds.cast_column("label", ClassLabel(names=label_names))


images, labels = _load_sorted_image_label_pairs()
n_train = int(len(images) * 0.9)
train_ds = _pairs_to_hf(images[:n_train], labels[:n_train])
val_ds = _pairs_to_hf(images[n_train:], labels[n_train:])

labels = list(train_ds.features["label"].names)
num_classes = len(labels)
id2label = {i: name for i, name in enumerate(labels)}
label2id = {name: i for i, name in enumerate(labels)}

print(f"Classes ({num_classes}): {labels}")


def preprocess_function(examples):
    images = [img.convert("RGB") for img in examples["image"]]
    # Fast image processors only support return_tensors="pt"; convert to NumPy for `datasets`.
    out = feature_extractor(images, return_tensors="pt")
    pixel_values = out["pixel_values"].detach().cpu().numpy()
    return {"pixel_values": pixel_values, "labels": examples["label"]}


train_ds = train_ds.map(preprocess_function, batched=True, remove_columns=train_ds.column_names)
val_ds = val_ds.map(preprocess_function, batched=True, remove_columns=val_ds.column_names)

Casting the dataset: 100%|██████████| 367/367 [00:00<00:00, 9403.01 examples/s]


Classes (5): ['dandelion', 'daisy', 'tulips', 'sunflowers', 'roses']


Map: 100%|██████████| 367/367 [00:03<00:00, 112.76 examples/s]


## Load pretrained `MobileViTForImageClassification`

Load ImageNet-pretrained weights and **replace the classifier head** for the flowers dataset (`num_labels=num_classes`, `ignore_mismatched_sizes=True`).



## Fine-tune with Hugging Face `Trainer`

We use a modest learning rate and weight decay typical for ViT-style fine-tuning.


In [5]:
model = MobileViTForImageClassification.from_pretrained(
    "apple/mobilevit-small",
    num_labels=num_classes,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable:,}")


Some weights of MobileViTForImageClassification were not initialized from the model checkpoint at apple/mobilevit-small and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([1000, 640]) in the checkpoint and torch.Size([5, 640]) in the model instantiated
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([5]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainable parameters: 4,940,837


The next cell defines `compute_metrics` and builds `Trainer`. Install `evaluate` for the default path (`pip install evaluate`).


In [6]:
try:
    import evaluate

    accuracy_metric = evaluate.load("accuracy")

    def compute_metrics(eval_pred):
        predictions, labels = eval_pred
        predictions = np.argmax(predictions, axis=1)
        return accuracy_metric.compute(predictions=predictions, references=labels)

except ImportError:
    from sklearn.metrics import accuracy_score

    def compute_metrics(eval_pred):
        predictions, labels = eval_pred
        predictions = np.argmax(predictions, axis=1)
        return {"accuracy": float(accuracy_score(labels, predictions))}


_base_args = dict(
    output_dir="models/mobilevit-small-tf-flowers",
    remove_unused_columns=False,
    save_strategy="epoch",
    learning_rate=learning_rate,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=epochs,
    weight_decay=weight_decay,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=10,
    fp16=torch.cuda.is_available(),
)
try:
    training_args = TrainingArguments(**_base_args, eval_strategy="epoch")
except TypeError:
    training_args = TrainingArguments(**_base_args, evaluation_strategy="epoch")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)


## Train (fine-tune)


In [7]:
trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy
1,0.842400,0.869672,0.762943
2,0.510600,0.588292,0.787466
3,0.304500,0.448238,0.839237
4,0.259200,0.453019,0.850136
5,0.323000,0.534950,0.798365


TrainOutput(global_step=1035, training_loss=0.5361916747070165, metrics={'train_runtime': 2639.817, 'train_samples_per_second': 6.256, 'train_steps_per_second': 0.392, 'total_flos': 9.625682673598464e+16, 'train_loss': 0.5361916747070165, 'epoch': 5.0})

## Evaluate


In [8]:
metrics = trainer.evaluate()
print(metrics)


{'eval_loss': 0.4530191421508789, 'eval_accuracy': 0.8501362397820164, 'eval_runtime': 47.6435, 'eval_samples_per_second': 7.703, 'eval_steps_per_second': 0.483, 'epoch': 5.0}


## Save the fine-tuned model

This PyTorch workflow saves weights and config in Hugging Face format.


In [9]:
save_dir = "models/mobilevit-small-tf-flowers-final"
trainer.save_model(save_dir)
feature_extractor.save_pretrained(save_dir)
print(f"Saved to {save_dir}")


Saved to ./mobilevit-small-tf-flowers-final


Hub checkpoint used for fine-tuning: [apple/mobilevit-small](https://huggingface.co/apple/mobilevit-small).